# Simulacion dinamica: el test deja de ser una foto fija

Hasta ahora el test media una asignacion **fija**: se decidia donde va cada producto y se
calculaba la distancia de cada batch como si el deposito nunca cambiara. Los eventos de
reabastecimiento se leian y no se usaban.

Aca el test pasa a ser una **simulacion**. Se parte de un estado del deposito (que producto
hay en cada ubicacion y con cuantas unidades) y se procesan los eventos en orden temporal:

- **Picks**: bajan stock. Si un producto esta en varias ubicaciones se va a la de **menor
  stock que alcance**; si ninguna alcanza sola, se junta de varias de menor a mayor. La
  ruta del batch se calcula sobre las ubicaciones **realmente visitadas**.
- **Reabastecimientos**: momento, producto y cantidad se replican del dato. Lo unico que se
  decide es **a donde van las unidades**, y esa decision la toma una **politica**
  intercambiable. Ahi es donde pesa el algoritmo de slotting.

Tres politicas:

| Politica | Que hace | Que modela |
|---|---|---|
| `replay` | recarga donde esta; si el stock llego a 0, muda a un hueco libre al azar | deposito sin plan: el WMS pone donde hay lugar |
| `home` | devuelve el producto a la ubicacion que le asigno el plan | WMS que conoce el plan y lo mantiene |
| `objective_greedy` | elige la ubicacion que minimiza el costo marginal del objetivo, con el estado actual | el algoritmo decidiendo online |


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from abs_affinity_based_slotting.config import RAW_DIR
from abs_affinity_based_slotting.data import WarehouseDataLoader, split_picking_events
from abs_affinity_based_slotting.demand import (build_cooccurrence, build_sku_demand,
    affinity_registry, filter_registry)
from abs_affinity_based_slotting.warehouse import (occupied_locations,
    build_location_costs, build_bay_distance_matrix)
from abs_affinity_based_slotting.slotting import build_instance
from abs_affinity_based_slotting.methods import CurrentSlotting, DemandGreedySlotting
from abs_affinity_based_slotting.evaluation import Evaluator
from abs_affinity_based_slotting.simulation import (
    build_event_stream, split_replenishments, state_from_initial_stock,
    advance_with_history, reslot_state, Simulator, infer_sku_capacity,
    ReplayGeneratorPolicy, HomePolicy, ObjectiveGreedyPolicy)

ds = WarehouseDataLoader(RAW_DIR).load_all()
split = split_picking_events(ds.picking_events, test_size=0.2)
universe = occupied_locations(ds.initial_stock)["sku"].to_numpy()

co = build_cooccurrence(split.train, skus=universe)
A = filter_registry.get("top_k")(k=10).filter(
    affinity_registry.get("jaccard")().build(co.matrix, co.support, co.n_batches))
instance = build_instance(build_sku_demand(split.train),
    build_location_costs(ds.initial_stock, ds.distances),
    build_bay_distance_matrix(ds.distances),
    initial_stock=ds.initial_stock, skus=universe, affinity=A)
instance